# Encoder failures versus Elevation and Azimuth
- Updated on 2025.12.02 by bquint

* Looks for failures/faults that indicate that we have encoder reading problems
* For each of these events, query the elevation and azimuth position
* Create a histogram to show where we have the most frequent events.

## Associated Tickets
- [SITCOM-2326](https://ls.st/sitcom-2326)

## Setup Notebook

In [ ]:
DAY_OBS_START = 20251115
DAY_OBS_END = 20251201
TIME_WINDOW_FOR_TELEMETRY = "10s"

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd

from astropy.time import Time, TimeDelta
from bokeh.io import output_file, reset_output  # Add reset_output here
from bokeh import layouts
from bokeh.models import HoverTool
from bokeh.plotting import figure, show, output_notebook, save

from lsst.summit.utils.dateTime import getDayObsEndTime, getDayObsStartTime
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient

import os
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Let's gather a client to query the EFD
efd_client = makeEfdClient()

# Get a time range based on the day obs
start_time = getDayObsStartTime(DAY_OBS_START)
end_time = getDayObsEndTime(DAY_OBS_END)

# Make sure it appears in the jupyter notebook
output_notebook()

# Create a placeholder for plots
os.makedirs("./plots", exist_ok=True)

## Find failure messages

When I started this analysis, I did not have idea about what messages to look for.  
I am keeping the code below to remind myself why I picked the codes below.

In [ ]:
df_warning = getEfdData(
    efd_client,
    topic="lsst.sal.MTMount.logevent_warning",
    columns="*",
    begin=start_time,
    end=end_time
)

In [ ]:
df_warning_subset = df_warning.drop_duplicates(subset="code")

for idx, row in df_warning_subset.iterrows():
    if "EIB" in row.text:
        print(f"Error code: {row.code} at {idx}\n{row.text}\n")
    elif "encoder" in row.text:
        print(f"Error code: {row.code} at {idx}\n{row.text}\n")

<br><hr>
It is important to notice that we **did not have any message saying**  
that we missed **Elevation encoder heads** in any time between `2025-04-15` and `2025-12-04`.  
<br>
For Azimuth encoder warnings, the codes we are interested in are `714`, `715`, `716`, and `717`.

In [ ]:
print(f"Dataframe size prior to filtering: {df_warning.index.size}")

mask = df_warning.code.isin([714, 715, 716, 717])
df_warning = df_warning[mask]
print(f"Dataframe size after filtering using the error codes: {df_warning.index.size}")

mask = df_warning.active
df_warning = df_warning[mask]
print(f"Dataframe size after filtering considering only `active` warnings: {df_warning.index.size}")

In [ ]:
df_warning.sort_index()

Cool.  
Now we can start extracting the telemetry for each failure or warning message. 

## Query Az/El Telemetry

In [ ]:
def query_tma_position(row, topic, delta_t):
    
    _df = getEfdData(
        client=efd_client,
        topic=topic,
        columns="actualPosition",
        begin=Time(row.name) - TimeDelta(delta_t),
        end=Time(row.name) + TimeDelta(delta_t),
        warn=False
    )

    try: 
        pos = _df['actualPosition'].mean()
    except KeyError: 
        print(f"Failed to collect data near {row.name}")
        pos = None

    return pos


# Keep a safe copy - easier for active development
df = df_warning.copy()  

# Update telescope telemetries
df['azimuth'] = df.apply(query_tma_position, 
    axis=1, topic="lsst.sal.MTMount.azimuth", delta_t=TIME_WINDOW_FOR_TELEMETRY)
df['elevation'] = df.apply(query_tma_position, 
    axis=1, topic="lsst.sal.MTMount.elevation", delta_t=TIME_WINDOW_FOR_TELEMETRY)
    
# Remove rows that have bad data
df = df.dropna(subset=["azimuth", "elevation"], how="any")

# Normalize azimuth to [-180, +180] range
df['azimuth'] = ((df['azimuth'] + 180) % 360) - 180

# Extract which encoder raised the warning
df['az_encoder_num'] = df['text'].str.extract(r'The azimuth encoder head (\d+)').astype(int)

# Keep only interesting columns
columns = ["code",  "azimuth", "elevation", "az_encoder_num"]
df = df.loc[:, columns]

print(f"Dataframe size containing useful telemetry: {df.index.size}")

## Plot histograms

Matplotlib is easier to record a PNG file.  
The bokeh version below is useful is you want more interactivity.

In [ ]:
%matplotlib inline
fig, (ax1, ax2) = plt.subplots(num="histograms", ncols=2, figsize=(12, 5))

ax1.hist(df['azimuth'], bins=360//5, fc="C0", ec="white")
ax1.set_xlabel("Azimuth [deg]")
ax1.set_ylabel("N-Faults")
ax1.grid(":", alpha=0.2)

ax2.hist(df['elevation'], bins=90//5, fc="C1", ec="white")
ax2.set_xlabel("Elevation [deg]")
ax2.set_ylabel("N-Faults")
ax2.grid(":", alpha=0.2)

fig.suptitle(f"Encoder Error Frequency at different angles\nStart: {start_time}, end: {end_time}")
fig.tight_layout()
plt.savefig(f"./plots/encoder_faults_vs_azel_{DAY_OBS_START}_{DAY_OBS_END}.png")
plt.show()

In [ ]:
output_notebook()

# Compute histogram data manually for azimuth
az_hist, az_edges = np.histogram(df['azimuth'], bins=360//5)
az_centers = (az_edges[:-1] + az_edges[1:]) / 2
az_width = az_edges[1] - az_edges[0]

# Compute histogram data manually for elevation
el_hist, el_edges = np.histogram(df['elevation'], bins=90//5)
el_centers = (el_edges[:-1] + el_edges[1:]) / 2
el_width = el_edges[1] - el_edges[0]

# Create azimuth plot
p1 = figure(
    width=550, 
    height=400,
    title="Azimuth",
    x_axis_label="Azimuth [deg]",
    y_axis_label="N-Faults",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

az_source = {'center': az_centers, 'count': az_hist, 'width': [az_width]*len(az_centers)}
p1.vbar(
    x='center', 
    top='count', 
    width='width', 
    source=az_source,
    color='#1f77b4',
    line_color='white'
)

hover1 = HoverTool(tooltips=[
    ('Azimuth', '@center{0.1f}°'),
    ('N-Faults', '@count')
])
p1.add_tools(hover1)
p1.grid.grid_line_alpha = 0.3

# Create elevation plot
p2 = figure(
    width=550, 
    height=400,
    title="Elevation",
    x_axis_label="Elevation [deg]",
    y_axis_label="N-Faults",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

el_source = {'center': el_centers, 'count': el_hist, 'width': [el_width]*len(el_centers)}
p2.vbar(
    x='center', 
    top='count', 
    width='width', 
    source=el_source,
    color='#ff7f0e',
    line_color='white'
)

hover2 = HoverTool(tooltips=[
    ('Elevation', '@center{0.1f}°'),
    ('N-Faults', '@count')
])
p2.add_tools(hover2)
p2.grid.grid_line_alpha = 0.3

# Combine plots and show
layout = layouts.row(p1, p2)
show(layout)

# Optional: save to HTML
output_file(f"./plots/encoder_faults_vs_azel_{DAY_OBS_START}_{DAY_OBS_END}.html")
save(layout)

## Calculate encoder positions

In the past, the azimuth coordinate system implemented by Tekniker had an offset of ~ 3.5 deg.  
This was reported and fixed in the azimuth actual position.  
However, the fix for the same offset was never deployed in the encoder absolute positions.  
So, instead of querying the position of the encoders, we will apply the offset documented in the figure below.

<img src="./azimuth_encoder_relative_positions.png" alt="Azimuth Encoder Relative Positions" width="750">

In [ ]:
def update_az_encoder_position(row : pd.Series) -> float:
    """Update encoder positions using the relative position of the encoders."""

    relative_encoder_positions = {
        1: 35.25,
        2: 145.75,
        3: -145.75,
        4: -35.25
    }

    return row.azimuth + relative_encoder_positions[row.az_encoder_num]


# Add encoder position based on the offsets above
df["az_encoder_pos"] = df.apply(update_az_encoder_position, axis=1)

# Normalize azimuth to [-180, +180] range
df['az_encoder_pos'] = ((df['az_encoder_pos'] + 180) % 360) - 180

In [ ]:
df

## Some statistics

In [ ]:
# Get the counts
counts = df.groupby(by="az_encoder_num").count()

print(f"Number of warnings per azimuth encoder: {counts["azimuth"]}")

## Histogram separated by Azimuth Encoder

In [ ]:
fig, ax = plt.subplots(num=f"stacked_histograms_by_encoder", figsize=(7, 5))

# Prepare data for all encoders
data_to_plot = []
labels = []
colors = []

for encoder_num in sorted(df["az_encoder_num"].unique()):
    mask = df["az_encoder_num"] == encoder_num
    sub_df = df[mask]
    data_to_plot.append(sub_df["az_encoder_pos"])
    labels.append(f"Encoder {encoder_num}")
    colors.append(f"C{encoder_num - 1}")

# Plot stacked histogram
ax.hist(
    data_to_plot, 
    bins=360//5,
    label=labels,
    color=colors,
    stacked=True,
    alpha=0.8,
    ec="white"
)

ax.set_xlabel("Azimuth [deg]")
ax.set_ylabel("N-Faults")
ax.grid(":", alpha=0.2)
ax.legend()

fig.suptitle(f"Stacked Histogram w/ Encoder position per Encoder\n Start: {DAY_OBS_START}, End: {DAY_OBS_END}")
fig.tight_layout()
plt.savefig(f"./plots/stacked_histogram_by_encoder_warnings_vs_azimuth_{DAY_OBS_START}_{DAY_OBS_END}.png")
plt.show()

### Extra histogram plots

The histograms below are an attempt to answer the question:  
"Where should we look at on the azimuth encoder tape?"  
  
The answer is not clear yet.  

In [ ]:
# Clear any previous Bokeh state
reset_output()
output_notebook()

# Bokeh default colors
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# Create two figures side by side
p1 = figure(
    width=600, 
    height=500,
    title=f"Telescope Azimuth Position\nStart: {DAY_OBS_START}, End: {DAY_OBS_END}",
    x_axis_label="Telescope Azimuth [deg]",
    y_axis_label="N-Faults",
    toolbar_location="above"
)

p2 = figure(
    width=600, 
    height=500,
    title=f"Encoder Azimuth Position\nStart: {DAY_OBS_START}, End: {DAY_OBS_END}",
    x_axis_label="Encoder Azimuth [deg]",
    y_axis_label="N-Faults",
    toolbar_location="above"
)

from bokeh.models import ColumnDataSource

# Prepare bins
bins = 360//5

# Get histogram edges (same for all encoders)
_, edges1 = np.histogram(df["azimuth"], bins=bins)
_, edges2 = np.histogram(df["az_encoder_pos"], bins=bins)

# Calculate bin centers and half-widths
centers1 = (edges1[:-1] + edges1[1:]) / 2
half_width1 = (edges1[1:] - edges1[:-1]) / 2

centers2 = (edges2[:-1] + edges2[1:]) / 2
half_width2 = (edges2[1:] - edges2[:-1]) / 2

# Initialize cumulative bottoms for stacking
bottom1 = np.zeros(len(edges1) - 1)
bottom2 = np.zeros(len(edges2) - 1)

# Create stacked histogram for each encoder
for encoder_num in sorted(df["az_encoder_num"].unique()):
    mask = df["az_encoder_num"] == encoder_num
    sub_df = df[mask]
    
    # Histogram 1: Telescope azimuth (stacked)
    hist1, _ = np.histogram(sub_df["azimuth"], bins=edges1)
    
    # Calculate mean encoder azimuth for each bin in plot 1
    encoder_az_means = []
    for i in range(len(edges1) - 1):
        bin_mask = (sub_df["azimuth"] >= edges1[i]) & (sub_df["azimuth"] < edges1[i+1])
        if bin_mask.sum() > 0:
            encoder_az_means.append(sub_df[bin_mask]["az_encoder_pos"].mean())
        else:
            encoder_az_means.append(np.nan)
    
    source1 = ColumnDataSource(data=dict(
        top=bottom1 + hist1,
        bottom=bottom1.copy(),
        left=edges1[:-1],
        right=edges1[1:],
        center=centers1,
        half_width=half_width1,
        encoder=[encoder_num] * len(hist1),
        color=[colors[encoder_num - 1]] * len(hist1),
        height=hist1,
        encoder_az=encoder_az_means
    ))
    
    p1.quad(
        top='top',
        bottom='bottom',
        left='left',
        right='right',
        source=source1,
        fill_color=colors[encoder_num - 1],
        line_color="white",
        alpha=0.8,
        legend_label=f"Encoder {encoder_num}"
    )
    
    # Update bottom for next encoder
    bottom1 += hist1
    
    # Histogram 2: Encoder azimuth position (stacked)
    hist2, _ = np.histogram(sub_df["az_encoder_pos"], bins=edges2)
    
    # Calculate mean telescope azimuth for each bin in plot 2
    telescope_az_means = []
    for i in range(len(edges2) - 1):
        bin_mask = (sub_df["az_encoder_pos"] >= edges2[i]) & (sub_df["az_encoder_pos"] < edges2[i+1])
        if bin_mask.sum() > 0:
            telescope_az_means.append(sub_df[bin_mask]["azimuth"].mean())
        else:
            telescope_az_means.append(np.nan)
    
    source2 = ColumnDataSource(data=dict(
        top=bottom2 + hist2,
        bottom=bottom2.copy(),
        left=edges2[:-1],
        right=edges2[1:],
        center=centers2,
        half_width=half_width2,
        encoder=[encoder_num] * len(hist2),
        color=[colors[encoder_num - 1]] * len(hist2),
        height=hist2,
        telescope_az=telescope_az_means
    ))
    
    p2.quad(
        top='top',
        bottom='bottom',
        left='left',
        right='right',
        source=source2,
        fill_color=colors[encoder_num - 1],
        line_color="white",
        alpha=0.8,
        legend_label=f"Encoder {encoder_num}"
    )
    
    # Update bottom for next encoder
    bottom2 += hist2

# Style both plots
for p in [p1, p2]:
    p.legend.location = "top_right"
    p.legend.click_policy = "hide"
    p.grid.grid_line_color = "gray"
    p.grid.grid_line_alpha = 0.5
    p.grid.grid_line_dash = [1, 4]
    p.grid.grid_line_width = 0.5
    

# Add hover tools - showing center ± half-width
hover1 = HoverTool(tooltips="""
    <div>
        <div style="color: @color; font-weight: bold;">Encoder @encoder</div>
        <div>Telescope Az: @center{0.1f} ± @half_width{0.1f} deg</div>
        <div>Encoder Az: @encoder_az{0.1f} deg</div>
        <div>Count: @height</div>
    </div>
""")
p1.add_tools(hover1)

hover2 = HoverTool(tooltips="""
    <div>
        <div style="color: @color; font-weight: bold;">Encoder @encoder</div>
        <div>Telescope Az: @telescope_az{0.1f} deg</div>
        <div>Encoder Az: @center{0.1f} ± @half_width{0.1f} deg</div>
        <div>Count: @height</div>
    </div>
""")
p2.add_tools(hover2)

# Combine plots side by side
layout = layouts.row(p1, p2)
show(layout)

# Optional: save to HTML
output_file(f"./plots/stacked_hist_encoder_faults_vs_azimuth{DAY_OBS_START}_{DAY_OBS_END}.html")
save(layout)

## Histogram separated by elevation encoder

We haven't had any warnings between `2025-04-15` and `2025-12-02`.   
So, no plots.

## Timeline for azimuth encoder warnings

Create a Bokeh plot showing a guitar plot where X represents the time, Y contains the 1 to 4 encoders,  
and each point in the guitar plot represents a guitar event.  
  
We want this Bokeh plot to have a hover tool showing the azimuth encoder position, the telescope position, and the time stamp.  
Freddy is insterested in finding sequence of events.  
This plot can help us finding them.